In [1]:
import sys
sys.path.insert(0, '../..')

import ast
import json
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

Path('../../data/processed').mkdir(parents=True, exist_ok=True)

RAW  = '../../data/raw/'
PROC = '../../data/processed/'

print("✅ Imports ready")

✅ Imports ready


In [2]:
# Load All RAW FILES

metadata    = pd.read_csv(RAW + 'movies_metadata.csv', low_memory=False)
credits     = pd.read_csv(RAW + 'credits.csv')
keywords    = pd.read_csv(RAW + 'keywords.csv')
links       = pd.read_csv(RAW + 'links.csv')
links_small = pd.read_csv(RAW + 'links_small.csv')
ratings     = pd.read_csv(RAW + 'ratings_small.csv')

print("Raw shapes:")
for name, df in [
    ('metadata',    metadata),
    ('credits',     credits),
    ('keywords',    keywords),
    ('links',       links),
    ('links_small', links_small),
    ('ratings',     ratings),
]:
    print(f"  {name:<15} {df.shape}")

Raw shapes:
  metadata        (45466, 24)
  credits         (45476, 3)
  keywords        (46419, 2)
  links           (45843, 3)
  links_small     (9125, 3)
  ratings         (100004, 4)


In [3]:
# Clean movies_metadata

print(f"Before: {metadata.shape}")

# Step 1 — fix ID to numeric, drop rows where ID is invalid
metadata['id'] = pd.to_numeric(metadata['id'], errors='coerce')
metadata = metadata.dropna(subset=['id'])
metadata['id'] = metadata['id'].astype(int)
print(f"After dropping invalid IDs: {len(metadata):,} rows")

# Step 2 — remove adult content
metadata = metadata[metadata['adult'] == 'False']
print(f"After removing adult content: {len(metadata):,} rows")

# Step 3 — remove duplicates on ID
metadata = metadata.drop_duplicates(subset='id')
print(f"After removing duplicates: {len(metadata):,} rows")

# Step 4 — fix numeric columns
num_cols = ['budget', 'revenue', 'runtime',
            'vote_average', 'vote_count', 'popularity']
for col in num_cols:
    metadata[col] = pd.to_numeric(metadata[col], errors='coerce')

# Step 5 — replace 0 budget/revenue with NaN (0 = unknown not actual zero)
metadata['budget']  = metadata['budget'].replace(0, np.nan)
metadata['revenue'] = metadata['revenue'].replace(0, np.nan)

# Step 6 — fix release_date and extract year
metadata['release_date'] = pd.to_datetime(
    metadata['release_date'], errors='coerce')
metadata['year'] = metadata['release_date'].dt.year.astype('Int64')

# Step 7 — fill missing text fields with empty string
metadata['overview'] = metadata['overview'].fillna('')
metadata['tagline']  = metadata['tagline'].fillna('')

# Step 8 — keep only useful columns
keep_cols = [
    'id', 'title', 'original_title', 'overview', 'tagline',
    'genres', 'release_date', 'year', 'original_language',
    'budget', 'revenue', 'runtime', 'vote_average', 'vote_count',
    'popularity', 'production_companies', 'poster_path', 'imdb_id'
]
metadata = metadata[keep_cols]

print(f"\nFinal metadata shape: {metadata.shape}")
print(metadata.dtypes)
print(f"\nNulls remaining:")
print(metadata.isnull().sum()[metadata.isnull().sum() > 0])

Before: (45466, 24)
After dropping invalid IDs: 45,463 rows
After removing adult content: 45,454 rows
After removing duplicates: 45,424 rows

Final metadata shape: (45424, 18)
id                               int64
title                           object
original_title                  object
overview                        object
tagline                         object
genres                          object
release_date            datetime64[ns]
year                             Int64
original_language               object
budget                         float64
revenue                        float64
runtime                        float64
vote_average                   float64
vote_count                     float64
popularity                     float64
production_companies            object
poster_path                     object
imdb_id                         object
dtype: object

Nulls remaining:
title                       3
release_date               86
year                       86


In [4]:
# Clean credits

print(f"Before: {credits.shape}")

# Fix ID
credits['id'] = pd.to_numeric(credits['id'], errors='coerce')
credits = credits.dropna(subset=['id'])
credits['id'] = credits['id'].astype(int)

# Remove duplicates
credits = credits.drop_duplicates(subset='id')

# Validate cast and crew are parseable JSON lists
def is_valid_json_list(val):
    try:
        result = ast.literal_eval(str(val))
        return isinstance(result, list)
    except:
        return False

valid_cast = credits['cast'].apply(is_valid_json_list)
valid_crew = credits['crew'].apply(is_valid_json_list)
credits = credits[valid_cast & valid_crew]

print(f"Final credits shape: {credits.shape}")

Before: (45476, 3)
Final credits shape: (45432, 3)


In [5]:
# Clean keywords
print(f"Before: {keywords.shape}")

keywords['id'] = pd.to_numeric(keywords['id'], errors='coerce')
keywords = keywords.dropna(subset=['id'])
keywords['id'] = keywords['id'].astype(int)
keywords = keywords.drop_duplicates(subset='id')

def is_valid_json_list(val):
    try:
        result = ast.literal_eval(str(val))
        return isinstance(result, list)
    except:
        return False

keywords = keywords[keywords['keywords'].apply(is_valid_json_list)]

print(f"Final keywords shape: {keywords.shape}")

Before: (46419, 2)
Final keywords shape: (45432, 2)


In [6]:
# Clean Links
print(f"Before links: {links.shape}")
print(f"Before links_small: {links_small.shape}")

# links.csv
links['tmdbId'] = pd.to_numeric(links['tmdbId'], errors='coerce')
links = links.dropna(subset=['tmdbId'])
links['tmdbId'] = links['tmdbId'].astype(int)
links = links.drop_duplicates(subset='movieId')

# links_small.csv
links_small['tmdbId'] = pd.to_numeric(links_small['tmdbId'], errors='coerce')
links_small = links_small.dropna(subset=['tmdbId'])
links_small['tmdbId'] = links_small['tmdbId'].astype(int)
links_small = links_small.drop_duplicates(subset='movieId')

print(f"Final links shape: {links.shape}")
print(f"Final links_small shape: {links_small.shape}")


Before links: (45843, 3)
Before links_small: (9125, 3)
Final links shape: (45624, 3)
Final links_small shape: (9112, 3)


In [7]:
# Clean Ratings

print(f"Before: {ratings.shape}")

# Remove nulls
ratings = ratings.dropna()

# Remove duplicate user-movie pairs (keep last rating)
before = len(ratings)
ratings = ratings.drop_duplicates(
    subset=['userId', 'movieId'], keep='last')
print(f"Removed {before - len(ratings):,} duplicate ratings")

# Validate rating range 0.5–5.0
before = len(ratings)
ratings = ratings[ratings['rating'].between(0.5, 5.0)]
print(f"Removed {before - len(ratings):,} out-of-range ratings")

# Convert timestamp to datetime
ratings['timestamp'] = pd.to_datetime(
    ratings['timestamp'], unit='s')

print(f"Final ratings shape: {ratings.shape}")
print(ratings.dtypes)

Before: (100004, 4)
Removed 0 duplicate ratings
Removed 0 out-of-range ratings
Final ratings shape: (100004, 4)
userId                int64
movieId               int64
rating              float64
timestamp    datetime64[ns]
dtype: object


In [9]:
# Build Master Movie Table

# Start with metadata
master = metadata.copy()

# Join credits — extract top 3 cast + director
def extract_cast(cast_str, top_n=3):
    try:
        cast = ast.literal_eval(cast_str)
        return [c['name'] for c in cast[:top_n]]
    except:
        return []

def extract_director(crew_str):
    try:
        crew = ast.literal_eval(crew_str)
        directors = [c['name'] for c in crew
                     if c['job'] == 'Director']
        return directors[0] if directors else ''
    except:
        return ''

credits['cast_names']  = credits['cast'].apply(extract_cast)
credits['director']    = credits['crew'].apply(extract_director)

master = master.merge(
    credits[['id', 'cast_names', 'director']],
    on='id', how='left'
)

# Join keywords
def extract_keywords(kw_str, top_n=10):
    try:
        kws = ast.literal_eval(kw_str)
        return [k['name'] for k in kws[:top_n]]
    except:
        return []

keywords['keyword_list'] = keywords['keywords'].apply(extract_keywords)

master = master.merge(
    keywords[['id', 'keyword_list']],
    on='id', how='left'
)

# Join links to get movieId (for ratings join)
master = master.merge(
    links[['movieId', 'tmdbId']],
    left_on='id', right_on='tmdbId',
    how='left'
)

# Fill list columns with empty lists
master['cast_names']   = master['cast_names'].apply(
    lambda x: x if isinstance(x, list) else [])
master['keyword_list'] = master['keyword_list'].apply(
    lambda x: x if isinstance(x, list) else [])

# Parse genres to list
def parse_genres(genre_str):
    try:
        return [g['name'] for g in ast.literal_eval(genre_str)]
    except:
        return []

master['genres_list'] = master['genres'].apply(parse_genres)

print(f"Master table shape: {master.shape}")
print(f"Columns: {list(master.columns)}")
print(f"\nMovies with movieId (linkable to ratings): "
      f"{master['movieId'].notna().sum():,}")
print(f"Movies without movieId (metadata only): "
      f"{master['movieId'].isna().sum():,}")

Master table shape: (45454, 24)
Columns: ['id', 'title', 'original_title', 'overview', 'tagline', 'genres', 'release_date', 'year', 'original_language', 'budget', 'revenue', 'runtime', 'vote_average', 'vote_count', 'popularity', 'production_companies', 'poster_path', 'imdb_id', 'cast_names', 'director', 'keyword_list', 'movieId', 'tmdbId', 'genres_list']

Movies with movieId (linkable to ratings): 45,454
Movies without movieId (metadata only): 0


In [15]:
print("FINAL DATA QUALITY REPORT")
print("=" * 55)

def count_duplicates(df):
    """Exclude list columns — pandas cannot hash lists for duplicate check"""
    non_list_cols = [
        c for c in df.columns
        if not df[c].apply(lambda x: isinstance(x, list)).any()
    ]
    return df.duplicated(subset=non_list_cols).sum()

datasets = {
    'metadata_clean': master,
    'credits_clean':  credits,
    'keywords_clean': keywords,
    'links_clean':    links,
    'ratings_clean':  ratings,
}

for name, df in datasets.items():
    nulls = df.isnull().sum().sum()
    dups  = count_duplicates(df)

    print(f"\n{name}")
    print(f"  Shape      : {df.shape}")
    print(f"  Total nulls: {nulls:,}")
    print(f"  Duplicates : {dups:,}")

# ID alignment check
rated_movie_ids  = set(ratings['movieId'].unique())
master_movie_ids = set(master['movieId'].dropna().astype(int))
overlap          = rated_movie_ids & master_movie_ids

print(f"\nID ALIGNMENT AFTER CLEANING")
print(f"  Rated movies           : {len(rated_movie_ids):,}")
print(f"  Master table movies    : {len(master_movie_ids):,}")
print(f"  Overlap (usable)       : {len(overlap):,}")
print(f"  Coverage               : "
      f"{len(overlap)/len(rated_movie_ids)*100:.1f}%")

FINAL DATA QUALITY REPORT

metadata_clean
  Shape      : (45454, 24)
  Total nulls: 75,473
  Duplicates : 0

credits_clean
  Shape      : (45432, 5)
  Total nulls: 0
  Duplicates : 0

keywords_clean
  Shape      : (45432, 3)
  Total nulls: 0
  Duplicates : 0

links_clean
  Shape      : (45624, 3)
  Total nulls: 0
  Duplicates : 0

ratings_clean
  Shape      : (100004, 4)
  Total nulls: 0
  Duplicates : 0

ID ALIGNMENT AFTER CLEANING
  Rated movies           : 9,066
  Master table movies    : 45,454
  Overlap (usable)       : 9,025
  Coverage               : 99.5%


In [16]:
# Save all Cleaned Files

# Save all cleaned files
master.to_csv(PROC + 'movies_master.csv', index=False)
credits.to_csv(PROC + 'credits_cleaned.csv', index=False)
keywords.to_csv(PROC + 'keywords_cleaned.csv', index=False)
links.to_csv(PROC + 'links_cleaned.csv', index=False)
links_small.to_csv(PROC + 'links_small_cleaned.csv', index=False)
ratings.to_csv(PROC + 'ratings_cleaned.csv', index=False)

# Save cleaning report
report = {
    "movies_master":     {"rows": len(master),
                          "cols": len(master.columns)},
    "credits_cleaned":   {"rows": len(credits)},
    "keywords_cleaned":  {"rows": len(keywords)},
    "links_cleaned":     {"rows": len(links)},
    "ratings_cleaned":   {"rows": len(ratings),
                          "unique_users":
                              int(ratings['userId'].nunique()),
                          "unique_movies":
                              int(ratings['movieId'].nunique())},
    "id_coverage_pct":   round(
        len(overlap)/len(rated_movie_ids)*100, 1),
}

with open(PROC + 'cleaning_report.json', 'w') as f:
    json.dump(report, f, indent=2)

print("✅ All cleaned files saved to data/processed/")
print(json.dumps(report, indent=2))

✅ All cleaned files saved to data/processed/
{
  "movies_master": {
    "rows": 45454,
    "cols": 24
  },
  "credits_cleaned": {
    "rows": 45432
  },
  "keywords_cleaned": {
    "rows": 45432
  },
  "links_cleaned": {
    "rows": 45624
  },
  "ratings_cleaned": {
    "rows": 100004,
    "unique_users": 671,
    "unique_movies": 9066
  },
  "id_coverage_pct": 99.5
}


In [17]:
print("""
FROM NOW ON — NEVER LOAD FROM data/raw/
ALWAYS LOAD FROM data/processed/

╔═══════════════════════════════════════════════════════╗
║  FILE YOU LOAD                  WHAT IT IS            ║
╠═══════════════════════════════════════════════════════╣
║  movies_master.csv              main movie table      ║
║                                 metadata + cast +     ║
║                                 keywords + links      ║
║                                                       ║
║  ratings_cleaned.csv            clean ratings         ║
║                                 (dev, 100K rows)      ║
║                                                       ║
║  ratings.csv (raw)              full ratings          ║
║                                 (26M, PySpark only)   ║
╚═══════════════════════════════════════════════════════╝

Example:
  movies  = pd.read_csv('data/processed/movies_master.csv')
  ratings = pd.read_csv('data/processed/ratings_cleaned.csv')
""")


FROM NOW ON — NEVER LOAD FROM data/raw/
ALWAYS LOAD FROM data/processed/

╔═══════════════════════════════════════════════════════╗
║  FILE YOU LOAD                  WHAT IT IS            ║
╠═══════════════════════════════════════════════════════╣
║  movies_master.csv              main movie table      ║
║                                 metadata + cast +     ║
║                                 keywords + links      ║
║                                                       ║
║  ratings_cleaned.csv            clean ratings         ║
║                                 (dev, 100K rows)      ║
║                                                       ║
║  ratings.csv (raw)              full ratings          ║
║                                 (26M, PySpark only)   ║
╚═══════════════════════════════════════════════════════╝

Example:
  movies  = pd.read_csv('data/processed/movies_master.csv')
  ratings = pd.read_csv('data/processed/ratings_cleaned.csv')

